In [3]:
# getting the warriors team id
import nba_api
from nba_api.stats.static import teams

nba_teams = teams.get_teams()

# gonna go through the dictionary for the warriors and find the team id
warriors = [team for team in nba_teams if team["abbreviation"] == "GSW"][0]
warriors_id = warriors["id"]
print(f"warriors_id: {warriors_id}")

warriors_id: 1610612744


In [8]:
import sys
print(sys.executable)

/Users/hussaintaheri/Desktop/sports-win-predictor/venv/bin/python


In [5]:
# lets query for the last regular season game where the warriors are playing
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.library.parameters import Season, SeasonType

gamefinder = leaguegamefinder.LeagueGameFinder(
    team_id_nullable = warriors_id,
    season_nullable = Season.default,
    season_type_nullable = SeasonType.regular
)

games_dict = gamefinder.get_normalized_dict()
games = games_dict["LeagueGameFinderResults"]
game = games[0]
game_id = game["GAME_ID"]
game_matchup = game["MATCHUP"]

print(f"Searching through {len(games)} game(s) for the game_id of {game_id} where {game_matchup}")

Searching through 74 game(s) for the game_id of 0022501073 where GSW vs. WAS


In [6]:
# query for the play by play of the most recent gsw regular season game
from nba_api.stats.endpoints import playbyplayv3

df = playbyplayv3.PlayByPlayV3(game_id).get_data_frames()[0]
df

,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,...,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId
0,0022501073,2,PT12M00.00S,1,0,,0,,,0,...,0,0,0,,Start of 1st Period (10:11 PM EST),period,start,1,0,1
1,0022501073,4,PT12M00.00S,1,1610612744,GSW,204001,Porziņģis,K. Porziņģis,0,...,,,0,h,Jump Ball Porzingis vs. Sarr: Tip to Carrington,Jump Ball,,1,0,2
2,0022501073,7,PT11M43.00S,1,1610612764,WAS,1642267,Carrington,B. Carrington,108,...,0,2,2,v,Carrington 17' Pullup Jump Shot (2 PTS),Made Shot,Pullup Jump shot,1,2,3
3,0022501073,8,PT11M27.00S,1,1610612744,GSW,204001,Porziņģis,K. Porziņģis,141,...,3,2,5,h,Porzingis 27' 3PT Jump Shot (3 PTS) (Santos 1 ...,Made Shot,Jump Shot,1,3,4
4,0022501073,10,PT11M22.00S,1,1610612764,WAS,1642267,Carrington,B. Carrington,0,...,,,0,v,Carrington Out of Bounds - Bad Pass Turnover T...,Turnover,Out of Bounds - Bad Pass Turnover,1,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
468,0022501073,701,PT00M11.40S,4,1610612744,GSW,1630611,Santos,G. Santos,0,...,131,126,257,h,Santos Free Throw 2 of 2 (27 PTS),Free Throw,Free Throw 2 of 2,1,0,469
469,0022501073,702,PT00M11.40S,4,0,,1610612764,,,0,...,,,0,v,Wizards Timeout: Regular (Reg.7 Short 0),Timeout,Regular,0,0,470
470,0022501073,703,PT00M10.40S,4,1610612764,WAS,1630702,Hardy,J. Hardy,0,...,,,0,v,Hardy Bad Pass Turnover (P1.T11),Turnover,Bad Pass,1,0,471
471,0022501073,703,PT00M10.40S,4,1610612744,GSW,1641764,Podziemski,B. Podziemski,0,...,,,0,h,Podziemski STEAL (2 STL),,,1,0,472


In [7]:
# documentation for the leaguegamefinder parameters
help(leaguegamefinder)

Help on module nba_api.stats.endpoints.leaguegamefinder in nba_api.stats.endpoints:

NAME
    nba_api.stats.endpoints.leaguegamefinder - LeagueGameFinder endpoint for finding games based on various filters.

DESCRIPTION
    Example:
        >>> from nba_api.stats.endpoints import LeagueGameFinder
        >>> # Get Lakers games from 2023-24 season
        >>> games = LeagueGameFinder(
        ...     player_or_team_abbreviation='T',
        ...     team_id_nullable='1610612747',
        ...     season_nullable='2023-24'
        ... )
        >>> df = games.league_game_finder_results.get_data_frame()

    Important Notes:
        **Date Filtering (Issue #526):**
        The date_from_nullable and date_to_nullable parameters work correctly and
        filter games by date range. Use format 'YYYY-MM-DD'.

        Example - filter by date range:
            >>> # Get all games in January 2024
            >>> games = LeagueGameFinder(
            ...     player_or_team_abbreviation='T',
    

In [9]:
# create a csv file to append our data to
filename = "nba_pbp_2024_2025.csv"
with open(filename, "w", newline = "") as csvfile:
    pass
print(f"{filename} was created")

nba_pbp_2024_2025.csv was created


In [13]:
%%time
# getting every single team id and getting every game id and adding it to our set to avoid getting dups while also pulling the data 
# and using sleep to not get an error for spamming requests
import pandas as pd
import nba_api
from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.library.parameters import Season, SeasonType
from nba_api.stats.endpoints import playbyplayv3
import time

nba_teams = teams.get_teams()
games_set = set()
season_dfs = []
t_counter, g_counter = 0, 0

for team in nba_teams:
    try:
        team_id = team["id"]
        team_games = leaguegamefinder.LeagueGameFinder(
            team_id_nullable = team_id,
            season_nullable = "2024-25",
            season_type_nullable = SeasonType.regular
        )
        time.sleep(1)
        games_dict = team_games.get_normalized_dict()
        games = games_dict["LeagueGameFinderResults"]
    except Exception as e:
        print(f"Error: {e}, team"
        t_counter += 1
        continue

    for game in games:
        try:
            game_id = game["GAME_ID"]
            if game_id in games_set:
                continue
                
            games_set.add(game_id)
    
            df = playbyplayv3.PlayByPlayV3(game_id).get_data_frames()[0]
            time.sleep(1)
            season_dfs.append(df)
        except Exception as e:
            print(f"Error: {e}, game")
            g_counter += 1

Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Error for an game
Request Erro

In [14]:
t_counter, g_counter

(21, 44)

In [ ]:
season_df = pd.concat(season_dfs)
season_df.to_csv("nba_pbp_2024_2025.csv", "w", newline = "")